# Create Warren Alpert Foundation Prize Awards

Creates OpenAlex award rows from the Warren Alpert Foundation's official prize-recipient API.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/warren_alpert_prize_to_s3.py` to fetch the official JSON endpoint, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:**
- Prize page: `https://www.warrenalpert.org/prize/`
- Recipient archive: `https://www.warrenalpert.org/prize-recipients/`
- API endpoint: `https://www.warrenalpert.org/wp-json/winners/v1/info`

**S3 location:** `s3a://openalex-ingest/awards/warren_alpert_prize/warren_alpert_prize_recipients.parquet`

**Local full-source validation on 2026-05-26:** 91 official recipient rows from 37 cohorts, 1987-2025. Coverage: 100% title/year/landing-url/description/bio/amount/currency; 86/91 affiliation strings (94.5%). Amount is split per official rule: USD 500,000 per cohort, divided equally among recipients when there is more than one.

**OpenAlex funder mapping:**
- Chosen funder_id: 4320307125
- Display name: Warren Alpert Foundation
- DOI/ROR: not present on the OpenAlex funder row
- Rationale: the official prize page states the Warren Alpert Foundation funds the prize; Harvard Medical School administers it in concert with the foundation.

**Mapping summary:** One OpenAlex award row per official recipient. `lead_investigator` carries the recipient's name and first listed institutional location when available. `description` uses the cohort-level achievement description; `bio` remains in the raw table for QA but is not inserted as the award description to avoid mixing person biography with award citation.

## Step 1: Load Raw Parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.warren_alpert_prize_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/warren_alpert_prize/warren_alpert_prize_recipients.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-26 found 91 rows.
SELECT COUNT(*) as total_warren_alpert_recipients
FROM openalex.awards.warren_alpert_prize_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.warren_alpert_prize_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, cohort_label, recipient_name, amount, currency, affiliation_raw
FROM openalex.awards.warren_alpert_prize_raw
ORDER BY source_year DESC, recipient_position
LIMIT 10;

## Step 1.5: Source, Money, and Key Scans

In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'warren_alpert_prize_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(recipient_name) AS rows_with_recipient,
    COUNT(description) AS rows_with_description,
    COUNT(bio) AS rows_with_bio,
    COUNT(affiliation_raw) AS rows_with_affiliation,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    MIN(TRY_CAST(start_year_raw AS INT)) AS min_start_year,
    MAX(TRY_CAST(end_year_raw AS INT)) AS max_end_year,
    SUM(TRY_CAST(amount AS DOUBLE)) AS total_usd
FROM openalex.awards.warren_alpert_prize_raw;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_id) AS distinct_source_ids
FROM openalex.awards.warren_alpert_prize_raw;

In [ ]:
%sql
SELECT cohort_label, COUNT(*) AS rows, MIN(amount) AS per_recipient_amount, SUM(TRY_CAST(amount AS DOUBLE)) AS cohort_total
FROM openalex.awards.warren_alpert_prize_raw
GROUP BY cohort_label
ORDER BY TRY_CAST(REGEXP_EXTRACT(cohort_label, '(\d{4})$', 1) AS INT) DESC;

In [ ]:
%sql
SELECT currency, TRY_CAST(amount AS DOUBLE) AS amount, COUNT(*) AS rows
FROM openalex.awards.warren_alpert_prize_raw
GROUP BY currency, TRY_CAST(amount AS DOUBLE)
ORDER BY amount;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Warren Alpert Foundation F4320307125'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320307125;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320307125;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.warren_alpert_prize_awards
USING delta
AS
WITH
funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307125
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.warren_alpert_prize_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        NULLIF(TRIM(g.currency), '') as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'prize' as funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'Warren Alpert Foundation Prize') as funder_scheme,
        'warren_alpert_prize' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,
        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation_raw), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN funder_resolved f
)
SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 126

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'warren_alpert_prize' AND priority = 126;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    126 as priority
FROM openalex.awards.warren_alpert_prize_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_warren_alpert_awards
FROM openalex.awards.warren_alpert_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.warren_alpert_prize_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year, end_year,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       lead_investigator.affiliation.name AS affiliation
FROM openalex.awards.warren_alpert_prize_awards
ORDER BY start_year DESC, funder_award_id
LIMIT 10;

In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids,
       COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.warren_alpert_prize_awards;

In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) AS cnt
FROM openalex.awards.warren_alpert_prize_awards
GROUP BY funder.display_name, funder_id;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(description) as has_description, COUNT(amount) as has_amount,
       COUNT(currency) as has_currency, COUNT(lead_investigator) as has_lead,
       COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.warren_alpert_prize_awards;

In [ ]:
%sql
SELECT currency, amount, COUNT(*) AS rows
FROM openalex.awards.warren_alpert_prize_awards
GROUP BY currency, amount
ORDER BY amount;

In [ ]:
%sql
SELECT start_year, end_year, COUNT(*) AS rows
FROM openalex.awards.warren_alpert_prize_awards
GROUP BY start_year, end_year
ORDER BY start_year DESC;

In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'warren_alpert_prize' AND priority = 126
GROUP BY priority, provenance;